# Phase 7 — Dashboard Data Preparation

This notebook prepares clean, dashboard-ready datasets for the final Power BI report.

The dashboard summarizes the main business outputs developed throughout the project:

- merchandise revenue and revenue trends;
- customer repeat-purchase behavior;
- customer churn risk;
- expected revenue at risk;
- customer segments;
- major churn-model drivers;
- the 13-week revenue forecast.

The goal is to expose business-facing outputs rather than reproduce the full modeling workflow inside Power BI.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent

DASHBOARD_DATA_DIR = (
    PROJECT_ROOT
    / "dashboard"
    / "data"
)

DASHBOARD_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Dashboard data folder:",
    DASHBOARD_DATA_DIR,
)

print(
    "Folder exists:",
    DASHBOARD_DATA_DIR.exists(),
)

Dashboard data folder: d:\Projects\retail-customer-retention\dashboard\data
Folder exists: True


## 7.2 PostgreSQL Connection

The dashboard-ready datasets are generated from the cleaned PostgreSQL transaction table used throughout the project.

Keeping the dashboard extraction tied to the validated merchandise table ensures that the reported business metrics remain consistent with the earlier analytical phases.

In [3]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text


load_dotenv(
    PROJECT_ROOT / ".env"
)


DATABASE_URL = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:"
    f"{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:"
    f"{os.getenv('DB_PORT')}/"
    f"{os.getenv('DB_NAME')}"
)


engine = create_engine(
    DATABASE_URL
)


with engine.connect() as connection:
    transaction_count = connection.execute(
        text(
            """
            SELECT COUNT(*)
            FROM retail_transactions;
            """
        )
    ).scalar()


print(
    "PostgreSQL connection successful."
)

print(
    "retail_transactions rows:",
    f"{transaction_count:,}",
)

PostgreSQL connection successful.
retail_transactions rows: 776,614


## 7.3 Monthly Revenue and Business Activity

The first dashboard table summarizes merchandise sales at a monthly level.

It provides the time-series data needed for business-facing dashboard metrics such as:

- merchandise revenue;
- order volume;
- active customers;
- units sold;
- average order value.

December 2011 is a partial month because the source dataset ends on 9 December 2011.

In [4]:
monthly_business_query = text(
    """
    SELECT
        DATE_TRUNC('month', invoice_date)::date AS month,
        SUM(line_revenue) AS revenue,
        COUNT(DISTINCT invoice) AS orders,
        COUNT(DISTINCT customer_id) AS active_customers,
        SUM(quantity) AS units_sold,
        SUM(line_revenue)
            / NULLIF(COUNT(DISTINCT invoice), 0) AS avg_order_value
    FROM retail_transactions
    GROUP BY
        DATE_TRUNC('month', invoice_date)::date
    ORDER BY
        month;
    """
)


monthly_business = pd.read_sql(
    monthly_business_query,
    engine,
)


monthly_business["month"] = pd.to_datetime(
    monthly_business["month"]
)


monthly_business_path = (
    DASHBOARD_DATA_DIR
    / "monthly_business.csv"
)


monthly_business.to_csv(
    monthly_business_path,
    index=False,
)


print(
    "Rows:",
    len(monthly_business),
)

print(
    "Date range:",
    monthly_business["month"].min().date(),
    "to",
    monthly_business["month"].max().date(),
)

print(
    "Total revenue:",
    f"£{monthly_business['revenue'].sum():,.2f}",
)

print(
    "Total orders:",
    f"{monthly_business['orders'].sum():,}",
)

print(
    "Saved to:",
    monthly_business_path,
)


monthly_business

Rows: 25
Date range: 2009-12-01 to 2011-12-01
Total revenue: £17,072,852.29
Total orders: 36,629
Saved to: d:\Projects\retail-customer-retention\dashboard\data\monthly_business.csv


,month,revenue,orders,active_customers,units_sold,avg_order_value
0,2009-12-01,677916.07,1497,951,398489,452.849746
1,2010-01-01,537251.50,991,717,369907,542.130676
2,2010-02-01,497937.57,1093,771,371658,455.569597
3,2010-03-01,665973.67,1509,1051,501290,441.334440
4,2010-04-01,585127.43,1315,939,350030,444.963825
5,2010-05-01,592734.13,1365,965,384606,434.237458
6,2010-06-01,629540.45,1482,1035,389391,424.791127
7,2010-07-01,581487.96,1362,924,324414,426.936828
8,2010-08-01,594561.17,1276,910,452168,465.957030
9,2010-09-01,805544.81,1659,1136,566740,485.560464


## 7.4 First-Observed vs Repeat Orders by Month

Each customer's merchandise orders are ranked chronologically using `ROW_NUMBER()`.

- The customer's first observed order in the dataset is classified as **First Observed**.
- All later orders from that customer are classified as **Repeat**.

This supports the dashboard's repeat-purchasing view while keeping the definition consistent with the earlier SQL analysis.

Because the dataset begins in December 2009, a customer's first observed order is not necessarily their first-ever purchase with the retailer.

December 2011 is also a partial month because the dataset ends on 9 December 2011.

In [5]:
monthly_order_type_query = text(
    """
    WITH customer_orders AS (
        SELECT
            customer_id,
            invoice,
            MIN(invoice_date) AS order_date
        FROM retail_transactions
        GROUP BY
            customer_id,
            invoice
    ),

    ranked_orders AS (
        SELECT
            customer_id,
            invoice,
            order_date,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY order_date, invoice
            ) AS order_number
        FROM customer_orders
    )

    SELECT
        DATE_TRUNC('month', order_date)::date AS month,

        COUNT(*) FILTER (
            WHERE order_number = 1
        ) AS first_observed_orders,

        COUNT(*) FILTER (
            WHERE order_number > 1
        ) AS repeat_orders,

        COUNT(*) AS total_orders

    FROM ranked_orders

    GROUP BY
        DATE_TRUNC('month', order_date)::date

    ORDER BY
        month;
    """
)


monthly_order_type = pd.read_sql(
    monthly_order_type_query,
    engine,
)


monthly_order_type["month"] = pd.to_datetime(
    monthly_order_type["month"]
)


monthly_order_type["repeat_order_share"] = (
    monthly_order_type["repeat_orders"]
    / monthly_order_type["total_orders"]
)


monthly_order_type["first_observed_order_share"] = (
    monthly_order_type["first_observed_orders"]
    / monthly_order_type["total_orders"]
)


monthly_order_type_path = (
    DASHBOARD_DATA_DIR
    / "monthly_order_type.csv"
)


monthly_order_type.to_csv(
    monthly_order_type_path,
    index=False,
)


# Validation against the Phase 1 SQL results
total_orders = monthly_order_type[
    "total_orders"
].sum()

first_orders = monthly_order_type[
    "first_observed_orders"
].sum()

repeat_orders = monthly_order_type[
    "repeat_orders"
].sum()

overall_repeat_order_share = (
    repeat_orders
    / total_orders
)


print(
    "Total orders:",
    f"{total_orders:,}",
)

print(
    "First-observed orders:",
    f"{first_orders:,}",
)

print(
    "Repeat orders:",
    f"{repeat_orders:,}",
)

print(
    "Overall repeat-order share:",
    f"{overall_repeat_order_share:.2%}",
)

print(
    "Saved to:",
    monthly_order_type_path,
)


monthly_order_type

Total orders: 36,629
First-observed orders: 5,861
Repeat orders: 30,768
Overall repeat-order share: 84.00%
Saved to: d:\Projects\retail-customer-retention\dashboard\data\monthly_order_type.csv


,month,first_observed_orders,repeat_orders,total_orders,repeat_order_share,first_observed_order_share
0,2009-12-01,951,546,1497,0.364729,0.635271
1,2010-01-01,382,609,991,0.614531,0.385469
2,2010-02-01,375,718,1093,0.656908,0.343092
3,2010-03-01,440,1069,1509,0.708416,0.291584
4,2010-04-01,293,1022,1315,0.777186,0.222814
5,2010-05-01,254,1111,1365,0.813919,0.186081
6,2010-06-01,267,1215,1482,0.819838,0.180162
7,2010-07-01,185,1177,1362,0.864170,0.135830
8,2010-08-01,162,1114,1276,0.873041,0.126959
9,2010-09-01,239,1420,1659,0.855937,0.144063


## 7.5 Customer Revenue Concentration

Customer revenue concentration measures how much merchandise revenue is generated by the highest-value customers.

Customers are ranked by total historical merchandise revenue, and cumulative revenue shares are calculated for the top:

- 1%;
- 5%;
- 10%;
- 20%.

This table supports the dashboard's customer-value concentration visual and highlights the financial importance of retaining high-value customers.

In [6]:
customer_revenue_query = text(
    """
    SELECT
        customer_id,
        SUM(line_revenue) AS total_revenue
    FROM retail_transactions
    GROUP BY
        customer_id
    ORDER BY
        total_revenue DESC;
    """
)


customer_revenue = pd.read_sql(
    customer_revenue_query,
    engine,
)


customer_revenue = customer_revenue.sort_values(
    "total_revenue",
    ascending=False,
).reset_index(drop=True)


total_customers = len(
    customer_revenue
)

total_revenue = customer_revenue[
    "total_revenue"
].sum()


concentration_rows = []


for top_pct in [1, 5, 10, 20]:
    customer_count = int(
        np.ceil(
            total_customers
            * top_pct
            / 100
        )
    )

    selected_revenue = customer_revenue.loc[
        : customer_count - 1,
        "total_revenue",
    ].sum()

    revenue_share = (
        selected_revenue
        / total_revenue
    )

    concentration_rows.append(
        {
            "top_customer_pct": top_pct,
            "customer_count": customer_count,
            "revenue": selected_revenue,
            "revenue_share": revenue_share,
            "revenue_share_pct": (
                revenue_share * 100
            ),
        }
    )


revenue_concentration = pd.DataFrame(
    concentration_rows
)


revenue_concentration_path = (
    DASHBOARD_DATA_DIR
    / "revenue_concentration.csv"
)


revenue_concentration.to_csv(
    revenue_concentration_path,
    index=False,
)


print(
    "Total customers:",
    f"{total_customers:,}",
)

print(
    "Total revenue:",
    f"£{total_revenue:,.2f}",
)

print(
    "Saved to:",
    revenue_concentration_path,
)


revenue_concentration

Total customers: 5,861
Total revenue: £17,072,852.29
Saved to: d:\Projects\retail-customer-retention\dashboard\data\revenue_concentration.csv


,top_customer_pct,customer_count,revenue,revenue_share,revenue_share_pct
0,1,59,5478416.15,0.320885,32.088465
1,5,294,8883638.62,0.520337,52.033711
2,10,587,10913024.53,0.639203,63.920336
3,20,1173,13179970.82,0.771984,77.198412


## 7.6 Business Overview KPI Summary

This table contains the headline metrics used on the Business Overview dashboard page.

The KPIs are calculated from the complete cleaned merchandise dataset:

- total merchandise revenue;
- total merchandise orders;
- total distinct customers;
- repeat customer rate.

A repeat customer is defined as a customer with more than one observed merchandise order in the dataset.

The repeat customer rate is distinct from the repeat order share: **72.27% of customers are repeat customers**, while **84.00% of orders are repeat orders**.

In [7]:
kpi_query = text(
    """
    WITH customer_orders AS (
        SELECT
            customer_id,
            COUNT(DISTINCT invoice) AS order_count
        FROM retail_transactions
        GROUP BY
            customer_id
    ),

    customer_metrics AS (
        SELECT
            COUNT(*) AS total_customers,
            COUNT(*) FILTER (
                WHERE order_count > 1
            ) AS repeat_customers
        FROM customer_orders
    ),

    transaction_metrics AS (
        SELECT
            SUM(line_revenue) AS total_revenue,
            COUNT(DISTINCT invoice) AS total_orders
        FROM retail_transactions
    )

    SELECT
        t.total_revenue,
        t.total_orders,
        c.total_customers,
        c.repeat_customers,
        c.repeat_customers::numeric
            / NULLIF(c.total_customers, 0)
            AS repeat_customer_rate
    FROM transaction_metrics t
    CROSS JOIN customer_metrics c;
    """
)


kpi_summary = pd.read_sql(
    kpi_query,
    engine,
)


kpi_summary_path = (
    DASHBOARD_DATA_DIR
    / "kpi_summary.csv"
)


kpi_summary.to_csv(
    kpi_summary_path,
    index=False,
)


print(
    "Total merchandise revenue:",
    f"£{kpi_summary.loc[0, 'total_revenue']:,.2f}",
)

print(
    "Total orders:",
    f"{kpi_summary.loc[0, 'total_orders']:,}",
)

print(
    "Total customers:",
    f"{kpi_summary.loc[0, 'total_customers']:,}",
)

print(
    "Repeat customers:",
    f"{kpi_summary.loc[0, 'repeat_customers']:,}",
)

print(
    "Repeat customer rate:",
    f"{kpi_summary.loc[0, 'repeat_customer_rate']:.2%}",
)

print(
    "Saved to:",
    kpi_summary_path,
)


kpi_summary

Total merchandise revenue: £17,072,852.29
Total orders: 36,629
Total customers: 5,861
Repeat customers: 4,236
Repeat customer rate: 72.27%
Saved to: d:\Projects\retail-customer-retention\dashboard\data\kpi_summary.csv


,total_revenue,total_orders,total_customers,repeat_customers,repeat_customer_rate
0,17072852.29,36629,5861,4236,0.722744


## 7.7 Retention Risk Dashboard Data

The second dashboard page focuses on the September 2011 customer-retention snapshot.

The dashboard is designed to answer four business questions:

- which customers should be prioritized for retention;
- how much projected customer value is exposed to churn risk;
- which customer segments show the highest risk;
- which model features contribute most strongly to churn predictions.

The retention view uses the **1 September 2011 cutoff**, with a **180-day observation window** and a **90-day outcome horizon**.

Customer prioritization is based on **estimated revenue at risk**, calculated from calibrated churn probability and a projected 90-day customer-value proxy.

The exported dashboard tables will include:

- retention-risk KPI summary;
- customer-level retention priorities;
- RFM segment risk summary;
- churn-model feature importance;
- contact-budget prioritization results.

In [8]:
# Define the retention-risk snapshot.
retention_cutoff = pd.Timestamp("2011-09-01")
retention_observation_start = (
    retention_cutoff - pd.Timedelta(days=180)
)
retention_label_end = (
    retention_cutoff + pd.Timedelta(days=90)
)


retention_snapshot_query = text(
    """
    WITH observation AS (
        SELECT
            customer_id,
            MAX(invoice_date) AS last_purchase_date,
            COUNT(DISTINCT invoice) AS frequency,
            SUM(line_revenue) AS monetary_value,
            SUM(quantity) AS total_units,
            SUM(line_revenue)
                / COUNT(DISTINCT invoice) AS avg_order_value
        FROM retail_transactions
        WHERE invoice_date >= :observation_start
          AND invoice_date < :cutoff_date
        GROUP BY customer_id
    ),

    future_purchases AS (
        SELECT DISTINCT
            customer_id
        FROM retail_transactions
        WHERE invoice_date >= :cutoff_date
          AND invoice_date < :label_end
    )

    SELECT
        o.customer_id,

        CAST(:cutoff_date AS timestamp) AS cutoff_date,

        CAST(:cutoff_date AS date)
            - CAST(o.last_purchase_date AS date)
            AS recency_days,

        o.frequency,

        ROUND(
            o.monetary_value,
            2
        ) AS monetary_value,

        o.total_units,

        ROUND(
            o.avg_order_value,
            2
        ) AS avg_order_value,

        CASE
            WHEN f.customer_id IS NULL THEN 1
            ELSE 0
        END AS churn

    FROM observation AS o

    LEFT JOIN future_purchases AS f
        ON o.customer_id = f.customer_id

    ORDER BY
        o.customer_id;
    """
)


retention_params = {
    "observation_start": retention_observation_start,
    "cutoff_date": retention_cutoff,
    "label_end": retention_label_end,
}


with engine.connect() as connection:
    retention_snapshot = pd.read_sql(
        retention_snapshot_query,
        connection,
        params=retention_params,
    )


# Convert PostgreSQL numeric values to standard numeric dtypes.
numeric_columns = [
    "recency_days",
    "frequency",
    "monetary_value",
    "total_units",
    "avg_order_value",
    "churn",
]

retention_snapshot[numeric_columns] = (
    retention_snapshot[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)


print(
    "Observation window:",
    retention_observation_start.date(),
    "to",
    retention_cutoff.date(),
    "(exclusive)",
)

print(
    "Outcome window:",
    retention_cutoff.date(),
    "to",
    retention_label_end.date(),
    "(exclusive)",
)

print(
    "\nCustomers:",
    len(retention_snapshot),
)

print(
    "Duplicate customers:",
    retention_snapshot["customer_id"]
    .duplicated()
    .sum(),
)

print(
    "Missing values:",
    retention_snapshot
    .isna()
    .sum()
    .sum(),
)

print(
    "\nChurn distribution:"
)

print(
    retention_snapshot["churn"]
    .value_counts()
    .sort_index()
)

print(
    "\nChurn rate:",
    f"{retention_snapshot['churn'].mean():.2%}",
)

retention_snapshot.head()

Observation window: 2011-03-05 to 2011-09-01 (exclusive)
Outcome window: 2011-09-01 to 2011-11-30 (exclusive)

Customers: 2768
Duplicate customers: 0
Missing values: 0

Churn distribution:
churn
0    1680
1    1088
Name: count, dtype: int64

Churn rate: 39.31%


,customer_id,cutoff_date,recency_days,frequency,monetary_value,total_units,avg_order_value,churn
0,12347,2011-09-01,30,3,1603.68,956,534.56,0
1,12348,2011-09-01,149,1,327.00,268,327.00,0
2,12352,2011-09-01,163,2,240.66,126,120.33,0
3,12353,2011-09-01,105,1,89.00,20,89.00,1
4,12354,2011-09-01,133,1,1079.40,530,1079.40,1


### Retention Snapshot Validation

The 1 September 2011 retention snapshot contains **2,768 eligible customers** with no duplicate customer records or missing feature values.

Using the subsequent 90-day outcome window:

- **1,088 customers (39.31%)** made no qualifying merchandise purchase and are labeled as churned.
- **1,680 customers (60.69%)** made at least one qualifying purchase and are labeled as non-churned.

This snapshot is the final held-out customer population used for the retention-risk dashboard. All predictive features are calculated only from transactions before the 1 September 2011 cutoff, preventing future-information leakage.

### 7.7.1 RFM Segmentation for the Retention Snapshot

To make the retention-risk dashboard more interpretable, the September 2011 customer snapshot is segmented using the same **Recency, Frequency, and Monetary (RFM)** framework developed in Phase 3.

Each customer receives percentile-based scores from **1 to 5** for:

- **Recency:** more recent customers receive higher scores;
- **Frequency:** customers with more orders receive higher scores;
- **Monetary value:** customers with greater historical spend receive higher scores.

The scores are then mapped into six business-oriented customer segments:

- Champions
- Loyal Customers
- At Risk
- Needs Attention
- Promising
- Hibernating

The segmentation is descriptive rather than predictive. The observed 90-day churn outcome is used only to evaluate how customer behavior differs across the resulting segments.

In [9]:
# Recreate the Phase 3 RFM segmentation on the
# 1 September 2011 retention snapshot.

retention_rfm = retention_snapshot.copy()


# -----------------------------
# RFM percentile scores: 1 to 5
# -----------------------------

# Lower recency is better, so more recent customers
# receive higher R scores.
retention_rfm["R_score"] = np.ceil(
    retention_rfm["recency_days"]
    .rank(
        method="average",
        ascending=False,
        pct=True,
    )
    * 5
).astype(int)


# Higher frequency is better.
retention_rfm["F_score"] = np.ceil(
    retention_rfm["frequency"]
    .rank(
        method="average",
        ascending=True,
        pct=True,
    )
    * 5
).astype(int)


# Higher monetary value is better.
retention_rfm["M_score"] = np.ceil(
    retention_rfm["monetary_value"]
    .rank(
        method="average",
        ascending=True,
        pct=True,
    )
    * 5
).astype(int)


retention_rfm["RFM_score"] = (
    retention_rfm["R_score"]
    + retention_rfm["F_score"]
    + retention_rfm["M_score"]
)


# -----------------------------
# Business-oriented RFM segments
# -----------------------------

def assign_rfm_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    elif r >= 3 and f >= 3:
        return "Loyal Customers"

    elif r <= 2 and (f >= 3 or m >= 4):
        return "At Risk"

    elif r >= 4 and f <= 2:
        return "Promising"

    elif r <= 2 and f <= 2:
        return "Hibernating"

    else:
        return "Needs Attention"


retention_rfm["rfm_segment"] = (
    retention_rfm.apply(
        assign_rfm_segment,
        axis=1,
    )
)


# -----------------------------
# Validate segment distribution
# -----------------------------

segment_validation = (
    retention_rfm
    .groupby(
        "rfm_segment",
        as_index=False,
    )
    .agg(
        customers=("customer_id", "count"),
        churners=("churn", "sum"),
        churn_rate=("churn", "mean"),
        avg_recency_days=("recency_days", "mean"),
        avg_monetary_value=("monetary_value", "mean"),
    )
)

segment_validation["churn_rate_pct"] = (
    segment_validation["churn_rate"] * 100
)

segment_validation = (
    segment_validation
    .sort_values(
        "churn_rate_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "Segmented customers:",
    retention_rfm["customer_id"].nunique(),
)

print(
    "Unassigned customers:",
    retention_rfm["rfm_segment"]
    .isna()
    .sum(),
)

segment_validation[
    [
        "rfm_segment",
        "customers",
        "churners",
        "churn_rate_pct",
        "avg_recency_days",
        "avg_monetary_value",
    ]
]

Segmented customers: 2768
Unassigned customers: 0


,rfm_segment,customers,churners,churn_rate_pct,avg_recency_days,avg_monetary_value
0,Hibernating,750,436,58.133333,126.717333,252.906613
1,Needs Attention,217,118,54.377880,57.552995,332.641982
2,Promising,245,131,53.469388,21.081633,443.256082
3,At Risk,361,151,41.828255,106.213296,1022.678310
4,Loyal Customers,649,202,31.124807,37.517720,947.984453
5,Champions,546,50,9.157509,16.756410,3904.669231


#### Retention-Snapshot Segment Interpretation

The September 2011 RFM segmentation shows a clear relationship between customer engagement and subsequent 90-day churn.

- **Hibernating** customers have the highest observed churn rate at **58.13%**, with an average recency of about **127 days**.
- **Needs Attention** and **Promising** customers also show relatively high churn rates of **54.38%** and **53.47%**, respectively.
- **At Risk** customers have a churn rate of **41.83%**, while maintaining substantially higher historical monetary value than the most inactive segments.
- **Loyal Customers** show a lower churn rate of **31.12%**.
- **Champions** have the lowest observed churn rate at only **9.16%** and the highest average historical monetary value, at approximately **£3,904.67**.

The pattern supports the business interpretation of the RFM framework: customers with stronger recent, frequent, and high-value purchasing behavior tend to be substantially less likely to churn.

For retention prioritization, segment membership should not be used alone. The dashboard will combine behavioral segmentation with **model-estimated churn probability and customer value** to identify customers whose potential loss represents the greatest financial exposure.

### 7.7.2 Retention Segment Summary Export

The RFM results are aggregated into a dashboard-ready segment table for Power BI.

For each customer segment, the export includes:

- number of eligible customers;
- number of observed churners;
- observed 90-day churn rate;
- average recency;
- average historical monetary value;
- share of the September 2011 eligible customer population.

A fixed segment order is also included so that Power BI can display the segments consistently from strongest customer relationship to weakest.

In [10]:
# Build the dashboard-ready RFM segment summary.

segment_order = {
    "Champions": 1,
    "Loyal Customers": 2,
    "At Risk": 3,
    "Needs Attention": 4,
    "Promising": 5,
    "Hibernating": 6,
}


retention_segment_summary = (
    retention_rfm
    .groupby(
        "rfm_segment",
        as_index=False,
    )
    .agg(
        customers=("customer_id", "count"),
        churners=("churn", "sum"),
        churn_rate=("churn", "mean"),
        avg_recency_days=("recency_days", "mean"),
        avg_monetary_value=("monetary_value", "mean"),
    )
)


retention_segment_summary["churn_rate_pct"] = (
    retention_segment_summary["churn_rate"] * 100
)

retention_segment_summary["customer_share_pct"] = (
    retention_segment_summary["customers"]
    / retention_segment_summary["customers"].sum()
    * 100
)

retention_segment_summary["segment_order"] = (
    retention_segment_summary["rfm_segment"]
    .map(segment_order)
)


# Round dashboard-facing values.
retention_segment_summary["churn_rate_pct"] = (
    retention_segment_summary["churn_rate_pct"]
    .round(2)
)

retention_segment_summary["customer_share_pct"] = (
    retention_segment_summary["customer_share_pct"]
    .round(2)
)

retention_segment_summary["avg_recency_days"] = (
    retention_segment_summary["avg_recency_days"]
    .round(1)
)

retention_segment_summary["avg_monetary_value"] = (
    retention_segment_summary["avg_monetary_value"]
    .round(2)
)


retention_segment_summary = (
    retention_segment_summary[
        [
            "segment_order",
            "rfm_segment",
            "customers",
            "churners",
            "churn_rate_pct",
            "customer_share_pct",
            "avg_recency_days",
            "avg_monetary_value",
        ]
    ]
    .sort_values("segment_order")
    .reset_index(drop=True)
)


segment_output_path = (
    DASHBOARD_DATA_DIR
    / "retention_segment_summary.csv"
)

retention_segment_summary.to_csv(
    segment_output_path,
    index=False,
)


print(
    "Saved:",
    segment_output_path,
)

print(
    "Rows:",
    len(retention_segment_summary),
)

retention_segment_summary

Saved: d:\Projects\retail-customer-retention\dashboard\data\retention_segment_summary.csv
Rows: 6


,segment_order,rfm_segment,customers,churners,churn_rate_pct,customer_share_pct,avg_recency_days,avg_monetary_value
0,1,Champions,546,50,9.16,19.73,16.8,3904.67
1,2,Loyal Customers,649,202,31.12,23.45,37.5,947.98
2,3,At Risk,361,151,41.83,13.04,106.2,1022.68
3,4,Needs Attention,217,118,54.38,7.84,57.6,332.64
4,5,Promising,245,131,53.47,8.85,21.1,443.26
5,6,Hibernating,750,436,58.13,27.10,126.7,252.91


### 7.7.3 Customer-Level Retention Prioritization

The dashboard prioritizes customers using a financial-risk metric rather than churn probability alone.

For each eligible customer:

- the selected Logistic Regression model produces a churn-risk score;
- Platt scaling converts the raw model score into a calibrated churn probability;
- the customer's 180-day merchandise revenue is scaled to a 90-day value proxy;
- estimated revenue at risk is calculated as:

**Revenue at risk = calibrated churn probability × 90-day value proxy**

where:

**90-day value proxy = 180-day monetary value × 0.5**

Customers are then ranked by estimated revenue at risk.

The top **20% of the September 2011 eligible population** are labeled as **Priority Retention Customers** for the dashboard. This produces a campaign-sized priority group rather than implying that all other customers are free of churn risk.

Revenue at risk is a model-based prioritization estimate and should not be interpreted as guaranteed recoverable revenue or as the causal effect of a retention intervention.

In [11]:
# Rebuild the historical snapshots required for
# chronological model training and probability calibration.

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def build_dashboard_snapshot(cutoff_date):
    """
    Build a leakage-safe customer snapshot using:
    - 180-day observation window
    - 90-day future churn-label window
    """

    cutoff_date = pd.Timestamp(cutoff_date)
    observation_start = (
        cutoff_date - pd.Timedelta(days=180)
    )
    label_end = (
        cutoff_date + pd.Timedelta(days=90)
    )

    params = {
        "observation_start": observation_start,
        "cutoff_date": cutoff_date,
        "label_end": label_end,
    }

    with engine.connect() as connection:
        snapshot = pd.read_sql(
            retention_snapshot_query,
            connection,
            params=params,
        )

    numeric_columns = [
        "recency_days",
        "frequency",
        "monetary_value",
        "total_units",
        "avg_order_value",
        "churn",
    ]

    snapshot[numeric_columns] = (
        snapshot[numeric_columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    return snapshot


model_cutoffs = [
    "2010-09-01",
    "2010-12-01",
    "2011-03-01",
]

calibration_cutoff = "2011-06-01"


model_snapshots = {
    cutoff: build_dashboard_snapshot(cutoff)
    for cutoff in model_cutoffs
}

calibration_snapshot = (
    build_dashboard_snapshot(
        calibration_cutoff
    )
)


model_snapshot_summary = pd.DataFrame(
    [
        {
            "cutoff_date": cutoff,
            "customers": len(snapshot),
            "churn_rate_pct": (
                snapshot["churn"].mean() * 100
            ),
        }
        for cutoff, snapshot
        in {
            **model_snapshots,
            calibration_cutoff: calibration_snapshot,
        }.items()
    ]
)


model_snapshot_summary[
    "churn_rate_pct"
] = (
    model_snapshot_summary[
        "churn_rate_pct"
    ]
    .round(2)
)


print(
    "Base-model training rows:",
    sum(
        len(snapshot)
        for snapshot
        in model_snapshots.values()
    ),
)

print(
    "Calibration rows:",
    len(calibration_snapshot),
)

print(
    "Final dashboard/test rows:",
    len(retention_snapshot),
)

model_snapshot_summary

Base-model training rows: 9625
Calibration rows: 2658
Final dashboard/test rows: 2768


,cutoff_date,customers,churn_rate_pct
0,2010-09-01,2822,37.99
1,2010-12-01,3465,61.96
2,2011-03-01,3338,57.64
3,2011-06-01,2658,50.04


### 7.7.4 Recreate the Calibrated Churn Model

The retention dashboard uses the same chronological modeling and calibration approach developed in Phase 4.

The process is:

- train the base Logistic Regression model on the first three historical snapshots;
- apply the same `log1p` feature transformations and standardization used during model development;
- use the 1 June 2011 snapshot only to fit the probability-calibration mapping;
- apply the frozen calibrated model to the 1 September 2011 dashboard population.

Platt scaling is used because the raw churn probabilities were systematically too high in the later period.

The calibrated probabilities are validated against the Phase 4 results before they are used to calculate customer-level revenue at risk.

In [12]:
# Recreate the Phase 4 calibrated churn-probability pipeline.

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)


feature_columns = [
    "recency_days",
    "frequency",
    "monetary_value",
    "total_units",
    "avg_order_value",
]

log_features = [
    "frequency",
    "monetary_value",
    "total_units",
    "avg_order_value",
]


# -----------------------------
# 1. Base-model training data
# -----------------------------

model_training = pd.concat(
    [
        model_snapshots[cutoff]
        for cutoff in model_cutoffs
    ],
    ignore_index=True,
)

X_model_train = (
    model_training[feature_columns]
    .copy()
)

y_model_train = (
    model_training["churn"]
    .astype(int)
    .copy()
)


# -----------------------------
# 2. Calibration and test data
# -----------------------------

X_calibration = (
    calibration_snapshot[feature_columns]
    .copy()
)

y_calibration = (
    calibration_snapshot["churn"]
    .astype(int)
    .copy()
)

X_dashboard = (
    retention_snapshot[feature_columns]
    .copy()
)

y_dashboard = (
    retention_snapshot["churn"]
    .astype(int)
    .copy()
)


for dataframe in [
    X_model_train,
    X_calibration,
    X_dashboard,
]:
    for column in feature_columns:
        dataframe[column] = pd.to_numeric(
            dataframe[column]
        )


# -----------------------------
# 3. Phase 3 preprocessing
# -----------------------------

X_model_train_lr = X_model_train.copy()
X_calibration_lr = X_calibration.copy()
X_dashboard_lr = X_dashboard.copy()


for column in log_features:
    X_model_train_lr[column] = np.log1p(
        X_model_train_lr[column]
    )

    X_calibration_lr[column] = np.log1p(
        X_calibration_lr[column]
    )

    X_dashboard_lr[column] = np.log1p(
        X_dashboard_lr[column]
    )


base_scaler = StandardScaler()

X_model_train_scaled = (
    base_scaler.fit_transform(
        X_model_train_lr
    )
)

X_calibration_scaled = (
    base_scaler.transform(
        X_calibration_lr
    )
)

X_dashboard_scaled = (
    base_scaler.transform(
        X_dashboard_lr
    )
)


# -----------------------------
# 4. Train frozen base model
# -----------------------------

base_logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

base_logistic_model.fit(
    X_model_train_scaled,
    y_model_train,
)


raw_calibration_probability = (
    base_logistic_model.predict_proba(
        X_calibration_scaled
    )[:, 1]
)

raw_dashboard_probability = (
    base_logistic_model.predict_proba(
        X_dashboard_scaled
    )[:, 1]
)


# Decision-function scores are used as the
# single input to the Platt calibration model.
calibration_scores = (
    base_logistic_model.decision_function(
        X_calibration_scaled
    )
    .reshape(-1, 1)
)

dashboard_scores = (
    base_logistic_model.decision_function(
        X_dashboard_scaled
    )
    .reshape(-1, 1)
)


# -----------------------------
# 5. Platt scaling
# -----------------------------

platt_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

platt_model.fit(
    calibration_scores,
    y_calibration,
)


calibrated_dashboard_probability = (
    platt_model.predict_proba(
        dashboard_scores
    )[:, 1]
)


# -----------------------------
# 6. Validate against Phase 4
# -----------------------------

calibration_validation = pd.DataFrame(
    {
        "metric": [
            "Observed churn rate",
            "Raw mean probability",
            "Calibrated mean probability",
            "Raw Brier score",
            "Calibrated Brier score",
            "PR-AUC",
            "ROC-AUC",
            "Platt intercept",
            "Platt slope",
        ],
        "value": [
            y_dashboard.mean(),
            raw_dashboard_probability.mean(),
            calibrated_dashboard_probability.mean(),
            brier_score_loss(
                y_dashboard,
                raw_dashboard_probability,
            ),
            brier_score_loss(
                y_dashboard,
                calibrated_dashboard_probability,
            ),
            average_precision_score(
                y_dashboard,
                calibrated_dashboard_probability,
            ),
            roc_auc_score(
                y_dashboard,
                calibrated_dashboard_probability,
            ),
            platt_model.intercept_[0],
            platt_model.coef_[0][0],
        ],
    }
)


print(
    "Base-model training customers:",
    len(model_training),
)

print(
    "Calibration customers:",
    len(calibration_snapshot),
)

print(
    "Dashboard customers:",
    len(retention_snapshot),
)

print(
    "\nPlatt intercept:",
    round(platt_model.intercept_[0], 4),
)

print(
    "Platt slope:",
    round(platt_model.coef_[0][0], 4),
)

calibration_validation

Base-model training customers: 9625
Calibration customers: 2658
Dashboard customers: 2768

Platt intercept: -0.3233
Platt slope: 1.2403


,metric,value
0,Observed churn rate,0.393064
1,Raw mean probability,0.543908
2,Calibrated mean probability,0.492381
3,Raw Brier score,0.224402
4,Calibrated Brier score,0.212038
5,PR-AUC,0.584964
6,ROC-AUC,0.730828
7,Platt intercept,-0.323295
8,Platt slope,1.240262


#### Calibrated Model Validation — Interpretation

The dashboard pipeline successfully reproduces the Phase 4 churn-model calibration results.

On the September 2011 held-out population:

- the observed 90-day churn rate is **39.31%**;
- the base Logistic Regression model produces an average raw churn probability of **54.39%**;
- Platt scaling reduces the average predicted probability to **49.24%**;
- the Brier score improves from **0.2244** to **0.2120**, indicating better probability accuracy after calibration;
- ranking performance remains stable, with **PR-AUC = 0.585** and **ROC-AUC = 0.731**.

The fitted Platt calibration mapping has an intercept of **-0.3233** and a slope of **1.2403**, matching the original Phase 4 analysis.

Although calibration improves probability accuracy, the mean calibrated probability remains above the observed churn rate. This residual overprediction indicates temporal calibration drift and should be considered when interpreting customer-level risk estimates.

The calibrated probabilities are therefore used as **relative financial-prioritization inputs**, rather than as perfectly calibrated forecasts of individual customer behavior.

### 7.7.5 Revenue-at-Risk Calculation

Customer-level churn probability is combined with historical customer value to estimate the financial exposure associated with churn risk.

The 180-day monetary value is scaled to a 90-day horizon:

**90-day value proxy = 180-day monetary value × 0.5**

Estimated revenue at risk is then calculated as:

**Revenue at risk = calibrated churn probability × 90-day value proxy**

This metric is used for prioritization only. It is not a forecast of guaranteed lost revenue and does not estimate how much revenue a retention intervention would recover.

In [13]:
# Calculate customer-level projected value and revenue at risk.

retention_priority = retention_rfm.copy()


# Add the calibrated churn probability.
retention_priority["calibrated_churn_probability"] = (
    calibrated_dashboard_probability
)


# Scale the 180-day historical monetary value
# to a 90-day customer-value proxy.
retention_priority["projected_90d_value"] = (
    retention_priority["monetary_value"] * 0.5
)


# Probability-weighted financial exposure.
retention_priority["revenue_at_risk"] = (
    retention_priority["calibrated_churn_probability"]
    * retention_priority["projected_90d_value"]
)


# Aggregate validation metrics.
total_projected_90d_value = (
    retention_priority["projected_90d_value"].sum()
)

total_revenue_at_risk = (
    retention_priority["revenue_at_risk"].sum()
)

revenue_at_risk_share = (
    total_revenue_at_risk
    / total_projected_90d_value
)


print(
    "Projected 90-day value proxy:",
    f"£{total_projected_90d_value:,.2f}",
)

print(
    "Estimated revenue at risk:",
    f"£{total_revenue_at_risk:,.2f}",
)

print(
    "Revenue-at-risk share:",
    f"{revenue_at_risk_share:.2%}",
)


retention_priority[
    [
        "customer_id",
        "rfm_segment",
        "calibrated_churn_probability",
        "projected_90d_value",
        "revenue_at_risk",
    ]
].head()

Projected 90-day value proxy: £1,743,419.59
Estimated revenue at risk: £371,526.51
Revenue-at-risk share: 21.31%


,customer_id,rfm_segment,calibrated_churn_probability,projected_90d_value,revenue_at_risk
0,12347,Champions,0.282370,801.84,226.415624
1,12348,Hibernating,0.712435,163.50,116.483043
2,12352,At Risk,0.632430,120.33,76.100311
3,12353,Hibernating,0.770581,44.50,34.290876
4,12354,At Risk,0.618906,539.70,334.023313


#### Revenue-at-Risk Validation — Interpretation

The dashboard preparation reproduces the Phase 4 financial-prioritization results exactly.

Across the **2,768 eligible customers** in the September 2011 snapshot:

- the projected 90-day customer-value proxy totals **£1,743,419.59**;
- probability-weighted estimated revenue at risk totals **£371,526.51**;
- estimated revenue at risk therefore represents **21.31%** of the projected 90-day customer value.

The calculation combines each customer's historical value with their calibrated churn probability, so customers with the highest churn probability are not necessarily the customers with the greatest financial exposure.

This distinction is important for the dashboard: retention prioritization will rank customers by **estimated revenue at risk**, rather than by churn probability alone.

### 7.7.6 Priority Retention Customers

The dashboard defines a focused retention audience by ranking customers according to estimated revenue at risk.

A **20% contact budget** is used to create the priority group:

- customers are sorted from highest to lowest estimated revenue at risk;
- the top 20% of eligible customers are labeled as **Priority Retention Customers**;
- this group is intended to represent a realistic campaign-capacity constraint rather than a claim that only these customers are at risk.

Because the September 2011 snapshot contains **2,768 eligible customers**, the 20% priority group should contain **554 customers** after rounding up.

In [14]:
# Rank customers by estimated revenue at risk
# and identify the top 20% priority group.

priority_budget = 0.20

retention_priority = (
    retention_priority
    .sort_values(
        "revenue_at_risk",
        ascending=False,
    )
    .reset_index(drop=True)
)


priority_customer_count = int(
    np.ceil(
        len(retention_priority)
        * priority_budget
    )
)


retention_priority["priority_rank"] = (
    np.arange(
        1,
        len(retention_priority) + 1,
    )
)


retention_priority[
    "priority_retention_customer"
] = (
    retention_priority["priority_rank"]
    <= priority_customer_count
)


priority_summary = pd.DataFrame(
    {
        "metric": [
            "Eligible customers",
            "Priority budget",
            "Priority customers",
            "Priority share",
            "Priority revenue at risk",
            "Share of total revenue at risk captured",
        ],
        "value": [
            len(retention_priority),
            priority_budget,
            retention_priority[
                "priority_retention_customer"
            ].sum(),
            retention_priority[
                "priority_retention_customer"
            ].mean(),
            retention_priority.loc[
                retention_priority[
                    "priority_retention_customer"
                ],
                "revenue_at_risk",
            ].sum(),
            (
                retention_priority.loc[
                    retention_priority[
                        "priority_retention_customer"
                    ],
                    "revenue_at_risk",
                ].sum()
                / total_revenue_at_risk
            ),
        ],
    }
)


print(
    "Eligible customers:",
    len(retention_priority),
)

print(
    "Priority customers:",
    priority_customer_count,
)

print(
    "Actual priority share:",
    f"{retention_priority['priority_retention_customer'].mean():.2%}",
)

print(
    "Revenue-at-risk share captured:",
    f"""
{
    retention_priority.loc[
        retention_priority["priority_retention_customer"],
        "revenue_at_risk",
    ].sum()
    / total_revenue_at_risk
:.2%}
""".strip(),
)


retention_priority[
    [
        "priority_rank",
        "customer_id",
        "rfm_segment",
        "calibrated_churn_probability",
        "projected_90d_value",
        "revenue_at_risk",
        "priority_retention_customer",
    ]
].head(10)

Eligible customers: 2768
Priority customers: 554
Actual priority share: 20.01%
Revenue-at-risk share captured: 44.01%


,priority_rank,customer_id,rfm_segment,calibrated_churn_probability,projected_90d_value,revenue_at_risk,priority_retention_customer
0,1,15749,At Risk,0.386040,10767.950,4156.861798,True
1,2,15098,At Risk,0.155630,19958.250,3106.103710,True
2,3,12590,At Risk,0.432373,4670.630,2019.456462,True
3,4,12409,At Risk,0.441978,3103.540,1371.696030,True
4,5,12435,At Risk,0.547661,1989.495,1089.568183,True
5,6,13081,Loyal Customers,0.179270,5701.255,1022.061588,True
6,7,12753,Loyal Customers,0.280364,3639.890,1020.494272,True
7,8,12688,Promising,0.407481,2436.905,992.991667,True
8,9,12744,Loyal Customers,0.297694,3034.030,903.211275,True
9,10,12378,Promising,0.434840,2004.310,871.554297,True


#### Priority-Customer Validation — Interpretation

Applying a 20% contact budget to the September 2011 retention population selects **554 Priority Retention Customers** from **2,768 eligible customers**.

Because the customer count must be a whole number, the selected group represents **20.01%** of the eligible population after rounding up.

Despite containing only one-fifth of customers, this priority group captures **44.01% of total estimated revenue at risk**.

This concentration shows that financial exposure is not distributed evenly across customers. Ranking by estimated revenue at risk therefore allows a limited retention campaign to focus on customers whose potential loss represents the greatest modeled financial exposure.

### 7.7.7 Customer Priority Dashboard Export

The customer-level retention table combines behavioral segmentation, calibrated churn risk, customer value, and financial prioritization into a single Power BI dataset.

Each row represents one customer from the **1 September 2011 retention snapshot** and includes:

- RFM segment and purchasing-behavior features;
- calibrated 90-day churn probability;
- projected 90-day customer-value proxy;
- estimated revenue at risk;
- financial-risk priority rank;
- Priority Retention Customer flag based on the top 20% contact budget.

The realized 90-day churn outcome is retained only as a **retrospective evaluation field**. It was not used as an input when generating the customer's predicted risk or priority ranking.

In [15]:
# Create the customer-level retention dataset for Power BI.

retention_customer_priority = (
    retention_priority[
        [
            "customer_id",
            "cutoff_date",
            "rfm_segment",
            "R_score",
            "F_score",
            "M_score",
            "RFM_score",
            "recency_days",
            "frequency",
            "monetary_value",
            "total_units",
            "avg_order_value",
            "calibrated_churn_probability",
            "projected_90d_value",
            "revenue_at_risk",
            "priority_rank",
            "priority_retention_customer",
            "churn",
        ]
    ]
    .copy()
)


# Use a clearer dashboard-facing name for the realized outcome.
retention_customer_priority = (
    retention_customer_priority.rename(
        columns={
            "churn": "observed_churn_90d",
        }
    )
)


# Add a text label that will be convenient for
# filtering and visual formatting in Power BI.
retention_customer_priority[
    "priority_group"
] = np.where(
    retention_customer_priority[
        "priority_retention_customer"
    ],
    "Priority Retention Customer",
    "Other Eligible Customer",
)


# Round monetary and probability fields for export.
retention_customer_priority[
    "calibrated_churn_probability"
] = (
    retention_customer_priority[
        "calibrated_churn_probability"
    ]
    .round(6)
)

for column in [
    "monetary_value",
    "avg_order_value",
    "projected_90d_value",
    "revenue_at_risk",
]:
    retention_customer_priority[column] = (
        retention_customer_priority[column]
        .round(2)
    )


# Keep financial-risk ranking order in the exported file.
retention_customer_priority = (
    retention_customer_priority
    .sort_values("priority_rank")
    .reset_index(drop=True)
)


customer_priority_output_path = (
    DASHBOARD_DATA_DIR
    / "retention_customer_priority.csv"
)

retention_customer_priority.to_csv(
    customer_priority_output_path,
    index=False,
)


print(
    "Saved:",
    customer_priority_output_path,
)

print(
    "Rows:",
    len(retention_customer_priority),
)

print(
    "Unique customers:",
    retention_customer_priority[
        "customer_id"
    ].nunique(),
)

print(
    "Priority customers:",
    retention_customer_priority[
        "priority_retention_customer"
    ].sum(),
)

print(
    "Missing values:",
    retention_customer_priority
    .isna()
    .sum()
    .sum(),
)


retention_customer_priority.head(10)

Saved: d:\Projects\retail-customer-retention\dashboard\data\retention_customer_priority.csv
Rows: 2768
Unique customers: 2768
Priority customers: 554
Missing values: 0


,customer_id,cutoff_date,rfm_segment,R_score,F_score,M_score,RFM_score,recency_days,frequency,monetary_value,total_units,avg_order_value,calibrated_churn_probability,projected_90d_value,revenue_at_risk,priority_rank,priority_retention_customer,observed_churn_90d,priority_group
0,15749,2011-09-01,At Risk,1,2,5,8,136,1,21535.90,9014,21535.90,0.386040,10767.95,4156.86,1,True,1,Priority Retention Customer
1,15098,2011-09-01,At Risk,2,4,5,11,83,3,39916.50,121,13305.50,0.155630,19958.25,3106.10,2,True,1,Priority Retention Customer
2,12590,2011-09-01,At Risk,2,2,5,9,112,1,9341.26,4282,9341.26,0.432373,4670.63,2019.46,3,True,1,Priority Retention Customer
3,12409,2011-09-01,At Risk,2,2,5,9,83,1,6207.08,3142,6207.08,0.441978,3103.54,1371.70,4,True,0,Priority Retention Customer
4,12435,2011-09-01,At Risk,1,2,5,8,168,1,3978.99,879,3978.99,0.547661,1989.50,1089.57,5,True,0,Priority Retention Customer
5,13081,2011-09-01,Loyal Customers,3,4,5,12,44,3,11402.51,7935,3800.84,0.179270,5701.26,1022.06,6,True,0,Priority Retention Customer
6,12753,2011-09-01,Loyal Customers,4,3,5,12,36,2,7279.78,3834,3639.89,0.280364,3639.89,1020.49,7,True,0,Priority Retention Customer
7,12688,2011-09-01,Promising,5,2,5,12,14,1,4873.81,3028,4873.81,0.407481,2436.91,992.99,8,True,1,Priority Retention Customer
8,12744,2011-09-01,Loyal Customers,3,3,5,11,45,2,6068.06,3544,3034.03,0.297694,3034.03,903.21,9,True,0,Priority Retention Customer
9,12378,2011-09-01,Promising,4,2,5,11,30,1,4008.62,2529,4008.62,0.434840,2004.31,871.55,10,True,1,Priority Retention Customer


### 7.7.8 Retention-Risk KPI Summary

A compact KPI table is created for the headline cards on the Retention Risk dashboard page.

The KPIs summarize the September 2011 retention population and the financial-risk prioritization framework:

- eligible customers in the retention snapshot;
- Priority Retention Customers under the 20% contact budget;
- total estimated revenue at risk;
- estimated revenue-at-risk share of the projected 90-day customer value.

Additional supporting metrics are retained for validation and dashboard calculations.

In [16]:
# Create the Retention Risk headline KPI table for Power BI.

retention_kpis = pd.DataFrame(
    {
        "eligible_customers": [
            len(retention_customer_priority)
        ],

        "priority_customers": [
            retention_customer_priority[
                "priority_retention_customer"
            ].sum()
        ],

        "priority_customer_share": [
            retention_customer_priority[
                "priority_retention_customer"
            ].mean()
        ],

        "projected_90d_value": [
            total_projected_90d_value
        ],

        "estimated_revenue_at_risk": [
            total_revenue_at_risk
        ],

        "revenue_at_risk_share": [
            revenue_at_risk_share
        ],

        "avg_calibrated_churn_probability": [
            retention_customer_priority[
                "calibrated_churn_probability"
            ].mean()
        ],

        "observed_churn_rate": [
            retention_customer_priority[
                "observed_churn_90d"
            ].mean()
        ],
    }
)


# Round dashboard-facing values.
retention_kpis[
    [
        "projected_90d_value",
        "estimated_revenue_at_risk",
    ]
] = (
    retention_kpis[
        [
            "projected_90d_value",
            "estimated_revenue_at_risk",
        ]
    ]
    .round(2)
)

retention_kpis[
    [
        "priority_customer_share",
        "revenue_at_risk_share",
        "avg_calibrated_churn_probability",
        "observed_churn_rate",
    ]
] = (
    retention_kpis[
        [
            "priority_customer_share",
            "revenue_at_risk_share",
            "avg_calibrated_churn_probability",
            "observed_churn_rate",
        ]
    ]
    .round(6)
)


retention_kpi_output_path = (
    DASHBOARD_DATA_DIR
    / "retention_kpis.csv"
)

retention_kpis.to_csv(
    retention_kpi_output_path,
    index=False,
)


print(
    "Saved:",
    retention_kpi_output_path,
)

retention_kpis

Saved: d:\Projects\retail-customer-retention\dashboard\data\retention_kpis.csv


,eligible_customers,priority_customers,priority_customer_share,projected_90d_value,estimated_revenue_at_risk,revenue_at_risk_share,avg_calibrated_churn_probability,observed_churn_rate
0,2768,554,0.200145,1743419.6,371526.51,0.213102,0.492381,0.393064


### 7.7.9 Retention Targeting by Contact Budget

The retention analysis compares two ways of selecting customers when campaign capacity is limited:

- **financial-risk targeting**, which ranks customers by estimated revenue at risk;
- **probability-only targeting**, which ranks customers only by calibrated churn probability.

The comparison is evaluated at **5%, 10%, and 20% customer-contact budgets**.

For each budget, the analysis measures the share of total estimated financial exposure captured by each targeting strategy. This shows whether incorporating customer value improves retention prioritization compared with selecting customers solely because they have a high predicted probability of churn.

In [17]:
# Compare financial-risk targeting with
# probability-only targeting at several contact budgets.

contact_budgets = [
    0.05,
    0.10,
    0.20,
]


# Rank once using each targeting strategy.
financial_ranking = (
    retention_priority
    .sort_values(
        "revenue_at_risk",
        ascending=False,
    )
    .reset_index(drop=True)
)

probability_ranking = (
    retention_priority
    .sort_values(
        "calibrated_churn_probability",
        ascending=False,
    )
    .reset_index(drop=True)
)


budget_results = []


for budget in contact_budgets:

    customers_contacted = int(
        np.ceil(
            len(retention_priority)
            * budget
        )
    )


    # Financial-risk targeting:
    # contact customers with the greatest
    # estimated revenue at risk.
    financial_exposure_captured = (
        financial_ranking
        .head(customers_contacted)[
            "revenue_at_risk"
        ]
        .sum()
        / total_revenue_at_risk
    )


    # Probability-only targeting:
    # contact customers with the greatest
    # calibrated churn probability.
    probability_exposure_captured = (
        probability_ranking
        .head(customers_contacted)[
            "revenue_at_risk"
        ]
        .sum()
        / total_revenue_at_risk
    )


    exposure_multiplier = (
        financial_exposure_captured
        / probability_exposure_captured
    )


    budget_results.append(
        {
            "contact_budget_pct": (
                budget * 100
            ),
            "customers_contacted": (
                customers_contacted
            ),
            "financial_targeting_exposure_pct": (
                financial_exposure_captured
                * 100
            ),
            "probability_targeting_exposure_pct": (
                probability_exposure_captured
                * 100
            ),
            "financial_vs_probability_multiplier": (
                exposure_multiplier
            ),
        }
    )


contact_budget_comparison = pd.DataFrame(
    budget_results
)


contact_budget_comparison[
    [
        "contact_budget_pct",
        "financial_targeting_exposure_pct",
        "probability_targeting_exposure_pct",
        "financial_vs_probability_multiplier",
    ]
] = (
    contact_budget_comparison[
        [
            "contact_budget_pct",
            "financial_targeting_exposure_pct",
            "probability_targeting_exposure_pct",
            "financial_vs_probability_multiplier",
        ]
    ]
    .round(2)
)


contact_budget_comparison

,contact_budget_pct,customers_contacted,financial_targeting_exposure_pct,probability_targeting_exposure_pct,financial_vs_probability_multiplier
0,5.0,139,19.22,1.26,15.21
1,10.0,277,29.05,3.34,8.71
2,20.0,554,44.01,9.09,4.84


#### Contact-Budget Comparison — Interpretation

Financial-risk targeting consistently captures substantially more estimated revenue exposure than probability-only targeting under the same customer-contact budget.

At a **5% contact budget**, financial-risk targeting captures **19.22%** of total estimated revenue at risk, compared with only **1.26%** using churn probability alone.

At a **10% budget**, the comparison is **29.05% versus 3.34%**, while at a **20% budget** it is **44.01% versus 9.09%**.

This corresponds to approximately:

- **15.21×** more financial exposure captured at a 5% budget;
- **8.71×** more at a 10% budget;
- **4.84×** more at a 20% budget.

The result highlights the difference between identifying customers who are most likely to churn and identifying customers whose potential churn represents the greatest modeled financial exposure.

In [18]:
# Export the contact-budget comparison for Power BI.

contact_budget_output_path = (
    DASHBOARD_DATA_DIR
    / "contact_budget_comparison.csv"
)

contact_budget_comparison.to_csv(
    contact_budget_output_path,
    index=False,
)


print(
    "Saved:",
    contact_budget_output_path,
)

print(
    "Rows:",
    len(contact_budget_comparison),
)

contact_budget_comparison

Saved: d:\Projects\retail-customer-retention\dashboard\data\contact_budget_comparison.csv
Rows: 3


,contact_budget_pct,customers_contacted,financial_targeting_exposure_pct,probability_targeting_exposure_pct,financial_vs_probability_multiplier
0,5.0,139,19.22,1.26,15.21
1,10.0,277,29.05,3.34,8.71
2,20.0,554,44.01,9.09,4.84


### 7.7.10 Churn Driver Importance

The Retention Risk dashboard includes a global view of the behavioral features that contribute most strongly to the churn model's predictions.

Because several purchasing variables are highly correlated, individual Logistic Regression coefficients are not interpreted as independent business effects.

Instead, the dashboard uses the **correlation-aware SHAP importance results from Phase 4** to summarize the relative contribution of each model feature.

The importance values represent average absolute SHAP magnitude across customers. They indicate how strongly a feature influences model predictions overall, but they should not be interpreted as causal effects.

In [19]:
# Create the global churn-driver importance table
# using the validated Phase 4 SHAP results.

churn_driver_importance = pd.DataFrame(
    {
        "feature": [
            "Frequency",
            "Monetary Value",
            "Total Units",
            "Recency",
            "Average Order Value",
        ],
        "mean_abs_shap": [
            0.3088,
            0.2346,
            0.2118,
            0.1972,
            0.0576,
        ],
    }
)


# Rank features from strongest to weakest
# global contribution to model predictions.
churn_driver_importance["importance_rank"] = (
    churn_driver_importance[
        "mean_abs_shap"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)


churn_driver_importance = (
    churn_driver_importance
    .sort_values("importance_rank")
    .reset_index(drop=True)
)


churn_driver_output_path = (
    DASHBOARD_DATA_DIR
    / "churn_driver_importance.csv"
)

churn_driver_importance.to_csv(
    churn_driver_output_path,
    index=False,
)


print(
    "Saved:",
    churn_driver_output_path,
)

print(
    "Rows:",
    len(churn_driver_importance),
)

churn_driver_importance

Saved: d:\Projects\retail-customer-retention\dashboard\data\churn_driver_importance.csv
Rows: 5


,feature,mean_abs_shap,importance_rank
0,Frequency,0.3088,1
1,Monetary Value,0.2346,2
2,Total Units,0.2118,3
3,Recency,0.1972,4
4,Average Order Value,0.0576,5


### 7.7.11 Final Retention Dashboard Export Validation

Before loading the Retention Risk data into Power BI, all Page 2 dashboard exports are checked together.

The validation confirms that:

- every required CSV file exists;
- customer-level data contains the expected 2,768 eligible customers;
- the priority-retention group contains 554 customers;
- the segment summary contains all six RFM segments;
- the contact-budget comparison contains the three planned campaign budgets;
- the churn-driver table contains all five model features.

This provides a final consistency check between the analytical workflow and the dashboard-ready datasets.

In [20]:
# Final validation of all Retention Risk dashboard exports.

page2_files = {
    "Retention segment summary":
        DASHBOARD_DATA_DIR / "retention_segment_summary.csv",

    "Customer priority":
        DASHBOARD_DATA_DIR / "retention_customer_priority.csv",

    "Retention KPIs":
        DASHBOARD_DATA_DIR / "retention_kpis.csv",

    "Contact budget comparison":
        DASHBOARD_DATA_DIR / "contact_budget_comparison.csv",

    "Churn driver importance":
        DASHBOARD_DATA_DIR / "churn_driver_importance.csv",
}


# -----------------------------
# 1. Confirm all files exist
# -----------------------------

for name, path in page2_files.items():
    print(
        f"{name}:",
        "FOUND" if path.exists() else "MISSING",
    )

assert all(
    path.exists()
    for path in page2_files.values()
)


# -----------------------------
# 2. Reload exported CSV files
# -----------------------------

segment_check = pd.read_csv(
    page2_files["Retention segment summary"]
)

customer_check = pd.read_csv(
    page2_files["Customer priority"]
)

kpi_check = pd.read_csv(
    page2_files["Retention KPIs"]
)

budget_check = pd.read_csv(
    page2_files["Contact budget comparison"]
)

driver_check = pd.read_csv(
    page2_files["Churn driver importance"]
)


# -----------------------------
# 3. Validate expected structure
# -----------------------------

expected_segments = {
    "Champions",
    "Loyal Customers",
    "At Risk",
    "Needs Attention",
    "Promising",
    "Hibernating",
}

expected_budgets = {
    5.0,
    10.0,
    20.0,
}

expected_drivers = {
    "Frequency",
    "Monetary Value",
    "Total Units",
    "Recency",
    "Average Order Value",
}


assert len(customer_check) == 2768

assert (
    customer_check["customer_id"]
    .nunique()
    == 2768
)

assert (
    customer_check[
        "priority_retention_customer"
    ]
    .sum()
    == 554
)

assert set(
    segment_check["rfm_segment"]
) == expected_segments

assert set(
    budget_check["contact_budget_pct"]
) == expected_budgets

assert set(
    driver_check["feature"]
) == expected_drivers


# -----------------------------
# 4. Validate headline KPI values
# -----------------------------

assert (
    int(kpi_check.loc[0, "eligible_customers"])
    == 2768
)

assert (
    int(kpi_check.loc[0, "priority_customers"])
    == 554
)

assert np.isclose(
    kpi_check.loc[
        0,
        "projected_90d_value",
    ],
    1743419.59,
)

assert np.isclose(
    kpi_check.loc[
        0,
        "estimated_revenue_at_risk",
    ],
    371526.51,
)

assert np.isclose(
    kpi_check.loc[
        0,
        "revenue_at_risk_share",
    ],
    0.213102,
)


# -----------------------------
# 5. Final summary
# -----------------------------

validation_summary = pd.DataFrame(
    {
        "dataset": [
            "Retention segment summary",
            "Customer priority",
            "Retention KPIs",
            "Contact budget comparison",
            "Churn driver importance",
        ],
        "rows": [
            len(segment_check),
            len(customer_check),
            len(kpi_check),
            len(budget_check),
            len(driver_check),
        ],
        "status": [
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
        ],
    }
)


print(
    "\nAll Page 2 retention dashboard "
    "exports passed validation."
)

validation_summary

Retention segment summary: FOUND
Customer priority: FOUND
Retention KPIs: FOUND
Contact budget comparison: FOUND
Churn driver importance: FOUND



All Page 2 retention dashboard exports passed validation.


,dataset,rows,status
0,Retention segment summary,6,PASS
1,Customer priority,2768,PASS
2,Retention KPIs,1,PASS
3,Contact budget comparison,3,PASS
4,Churn driver importance,5,PASS


## 7.8 Revenue Forecast Dashboard Data

This section prepares the dashboard-ready datasets for the **Revenue Forecast** page.

The forecasting dashboard uses the validated Phase 5 weekly forecasting workflow:

- revenue is aggregated to complete **Friday-ending weeks (`W-FRI`)**;
- the historical forecasting series contains **105 complete weekly observations** from **11 December 2009 through 9 December 2011**;
- the selected model is the **52-week Seasonal Naive** benchmark;
- the final forecast covers **13 weeks**, from **16 December 2011 through 9 March 2012**;
- forecast uncertainty is represented using the **90th percentile of historical absolute backtest errors (±£60,478.76 per week)**.

The empirical error range is a practical planning range based on historical backtest performance. It is **not** a formal statistical confidence interval.

Dashboard exports will include:

1. the 105-week historical merchandise-revenue series;
2. the 13-week point forecast with empirical lower and upper error bounds;
3. the four-model forecasting comparison used to select the final model;
4. a compact KPI summary for the Revenue Forecast dashboard cards.

In [23]:
# Recreate the validated Phase 5 complete weekly revenue series
# and export it for the Revenue Forecast dashboard.

weekly_history_query = text(
    """
    SELECT
        invoice_date::date AS date,
        SUM(line_revenue) AS revenue
    FROM retail_transactions
    GROUP BY
        invoice_date::date
    ORDER BY
        date;
    """
)


daily_forecast_raw = pd.read_sql(
    weekly_history_query,
    engine,
)


daily_forecast_raw["date"] = pd.to_datetime(
    daily_forecast_raw["date"]
)

daily_forecast_raw["revenue"] = pd.to_numeric(
    daily_forecast_raw["revenue"]
)


# Build the complete daily calendar used in Phase 5.
# Days without qualifying merchandise purchases
# are represented as £0 revenue.
forecast_calendar = pd.date_range(
    start=daily_forecast_raw["date"].min(),
    end=daily_forecast_raw["date"].max(),
    freq="D",
)


daily_forecast_revenue = (
    daily_forecast_raw
    .set_index("date")
    .reindex(forecast_calendar)
)

daily_forecast_revenue.index.name = "date"

daily_forecast_revenue["revenue"] = (
    daily_forecast_revenue["revenue"]
    .fillna(0)
)

daily_forecast_revenue = (
    daily_forecast_revenue
    .reset_index()
)


# Aggregate to Saturday-Friday weeks,
# matching the Phase 5 W-FRI forecasting series.
weekly_revenue = (
    daily_forecast_revenue
    .set_index("date")
    .resample("W-FRI")
    .agg(
        revenue=("revenue", "sum"),
        calendar_days=("revenue", "size"),
    )
    .reset_index()
    .rename(
        columns={
            "date": "week_end",
        }
    )
)


# Remove incomplete edge weeks exactly as in Phase 5.
weekly_revenue = (
    weekly_revenue.loc[
        weekly_revenue["calendar_days"] == 7
    ]
    .copy()
    .reset_index(drop=True)
)


# Prepare the dashboard-facing export.
weekly_revenue_history = (
    weekly_revenue[
        [
            "week_end",
            "revenue",
        ]
    ]
    .rename(
        columns={
            "revenue": "actual_revenue",
        }
    )
    .copy()
)


weekly_revenue_history[
    "actual_revenue"
] = (
    weekly_revenue_history[
        "actual_revenue"
    ]
    .round(2)
)


weekly_history_output_path = (
    DASHBOARD_DATA_DIR
    / "weekly_revenue_history.csv"
)


weekly_revenue_history.to_csv(
    weekly_history_output_path,
    index=False,
)


# Validate against the completed Phase 5 series.
assert len(weekly_revenue_history) == 105

assert (
    weekly_revenue_history["week_end"].min()
    == pd.Timestamp("2009-12-11")
)

assert (
    weekly_revenue_history["week_end"].max()
    == pd.Timestamp("2011-12-09")
)


print(
    "Saved:",
    weekly_history_output_path,
)

print(
    "Rows:",
    len(weekly_revenue_history),
)

print(
    "Date range:",
    weekly_revenue_history["week_end"].min().date(),
    "to",
    weekly_revenue_history["week_end"].max().date(),
)

print(
    "Zero-revenue weeks:",
    int(
        weekly_revenue_history[
            "actual_revenue"
        ].eq(0).sum()
    ),
)


weekly_revenue_history.head()

Saved: d:\Projects\retail-customer-retention\dashboard\data\weekly_revenue_history.csv
Rows: 105
Date range: 2009-12-11 to 2011-12-09
Zero-revenue weeks: 2


,week_end,actual_revenue
0,2009-12-11,218234.86
1,2009-12-18,216018.88
2,2009-12-25,46580.03
3,2010-01-01,0.00
4,2010-01-08,142373.83


### 7.8.2 Final 13-Week Revenue Forecast

The dashboard forecast uses the **52-week Seasonal Naive model**, which achieved the strongest rolling-origin validation performance in Phase 5.

For each of the next 13 Friday-ending weeks, the point forecast is equal to merchandise revenue from the corresponding week **52 weeks earlier**.

The forecast covers:

- **16 December 2011 through 9 March 2012**;
- a **13-week planning horizon**;
- an empirical weekly error margin of **±£60,478.76**.

The lower and upper planning bounds are calculated by subtracting and adding this historical backtest error margin to each weekly point forecast. Lower bounds are clipped at £0 because negative merchandise revenue is not meaningful for this dashboard.

This range is based on the **90th percentile of historical absolute forecast errors** and should be interpreted as a practical planning range rather than a formal statistical confidence interval.

One forecast week — **30 December 2011** — receives a point forecast of **£0** because the corresponding week 52 weeks earlier also had zero qualifying merchandise revenue. This is a **Seasonal Naive rule artefact**, not a prediction that revenue will certainly be zero.

In [24]:
# Create the final 13-week Seasonal Naive forecast
# and empirical backtest-based error bounds.

forecast_horizon = 13
seasonal_lag_weeks = 52
empirical_error_margin = 60478.76


# Build the 13 future Friday-ending weeks.
forecast_dates = pd.date_range(
    start=(
        weekly_revenue_history["week_end"].max()
        + pd.Timedelta(weeks=1)
    ),
    periods=forecast_horizon,
    freq="W-FRI",
)


# Seasonal Naive:
# each forecast equals the revenue from the
# corresponding week 52 weeks earlier.
history_lookup = (
    weekly_revenue_history
    .set_index("week_end")["actual_revenue"]
)


forecast_values = []

for forecast_date in forecast_dates:

    source_week = (
        forecast_date
        - pd.Timedelta(weeks=seasonal_lag_weeks)
    )

    forecast_values.append(
        history_lookup.loc[source_week]
    )


weekly_revenue_forecast = pd.DataFrame(
    {
        "week_end": forecast_dates,
        "forecast_revenue": forecast_values,
    }
)


# Empirical planning range based on the
# 90th percentile of historical absolute
# backtest errors.
weekly_revenue_forecast[
    "error_margin"
] = empirical_error_margin


weekly_revenue_forecast[
    "lower_bound"
] = (
    weekly_revenue_forecast[
        "forecast_revenue"
    ]
    - empirical_error_margin
).clip(lower=0)


weekly_revenue_forecast[
    "upper_bound"
] = (
    weekly_revenue_forecast[
        "forecast_revenue"
    ]
    + empirical_error_margin
)


# Flag the known zero-forecast rule artefact
# for dashboard annotation.
weekly_revenue_forecast[
    "zero_forecast_rule_artefact"
] = (
    weekly_revenue_forecast[
        "forecast_revenue"
    ]
    .eq(0)
)


# Round dashboard-facing currency values.
currency_columns = [
    "forecast_revenue",
    "error_margin",
    "lower_bound",
    "upper_bound",
]

weekly_revenue_forecast[
    currency_columns
] = (
    weekly_revenue_forecast[
        currency_columns
    ]
    .round(2)
)


forecast_output_path = (
    DASHBOARD_DATA_DIR
    / "weekly_revenue_forecast.csv"
)


weekly_revenue_forecast.to_csv(
    forecast_output_path,
    index=False,
)


# -----------------------------
# Validation
# -----------------------------

assert len(weekly_revenue_forecast) == 13

assert (
    weekly_revenue_forecast["week_end"].min()
    == pd.Timestamp("2011-12-16")
)

assert (
    weekly_revenue_forecast["week_end"].max()
    == pd.Timestamp("2012-03-09")
)

assert np.isclose(
    weekly_revenue_forecast[
        "forecast_revenue"
    ].sum(),
    1433386.67,
)

assert (
    weekly_revenue_forecast[
        "zero_forecast_rule_artefact"
    ].sum()
    == 1
)


print(
    "Saved:",
    forecast_output_path,
)

print(
    "Rows:",
    len(weekly_revenue_forecast),
)

print(
    "Forecast period:",
    weekly_revenue_forecast[
        "week_end"
    ].min().date(),
    "to",
    weekly_revenue_forecast[
        "week_end"
    ].max().date(),
)

print(
    "13-week forecast total: "
    f"£{weekly_revenue_forecast['forecast_revenue'].sum():,.2f}"
)

print(
    "Average weekly forecast: "
    f"£{weekly_revenue_forecast['forecast_revenue'].mean():,.2f}"
)

print(
    "Empirical weekly error margin: "
    f"±£{empirical_error_margin:,.2f}"
)

print(
    "\nZero-forecast rule artefact:"
)

display(
    weekly_revenue_forecast.loc[
        weekly_revenue_forecast[
            "zero_forecast_rule_artefact"
        ]
    ]
)

weekly_revenue_forecast

Saved: d:\Projects\retail-customer-retention\dashboard\data\weekly_revenue_forecast.csv
Rows: 13
Forecast period: 2011-12-16 to 2012-03-09
13-week forecast total: £1,433,386.67
Average weekly forecast: £110,260.51
Empirical weekly error margin: ±£60,478.76

Zero-forecast rule artefact:


,week_end,forecast_revenue,error_margin,lower_bound,upper_bound,zero_forecast_rule_artefact
2,2011-12-30,0.0,60478.76,0.0,60478.76,True


,week_end,forecast_revenue,error_margin,lower_bound,upper_bound,zero_forecast_rule_artefact
0,2011-12-16,171530.20,60478.76,111051.44,232008.96,False
1,2011-12-23,52456.86,60478.76,0.00,112935.62,False
2,2011-12-30,0.00,60478.76,0.00,60478.76,True
3,2012-01-06,98072.37,60478.76,37593.61,158551.13,False
4,2012-01-13,161353.69,60478.76,100874.93,221832.45,False
5,2012-01-20,170910.27,60478.76,110431.51,231389.03,False
6,2012-01-27,107375.49,60478.76,46896.73,167854.25,False
7,2012-02-03,107615.80,60478.76,47137.04,168094.56,False
8,2012-02-10,85104.38,60478.76,24625.62,145583.14,False
9,2012-02-17,121145.24,60478.76,60666.48,181624.00,False


#### Forecast Result

The selected Seasonal Naive model forecasts **£1,433,386.67** in merchandise revenue over the next 13 weeks, equivalent to approximately **£110,260.51 per week** on average.

The weekly planning range applies an empirical error margin of **±£60,478.76**, derived from historical rolling-origin backtest errors. This reflects the level of weekly forecast uncertainty observed during model evaluation rather than a formal confidence interval.

The week ending **30 December 2011** receives a **£0 point forecast** because the corresponding week 52 weeks earlier also recorded zero qualifying merchandise revenue. The dashboard will explicitly flag this value as a **Seasonal Naive rule artefact rather than a certainty of zero future revenue**.

### 7.8.3 Forecast Model Comparison

Four forecasting approaches were evaluated using chronological rolling-origin validation across four 13-week folds.

The models are compared using:

- **MAE** — mean absolute error;
- **RMSE** — root mean squared error;
- **sMAPE** — symmetric mean absolute percentage error.

The **52-week Seasonal Naive** model achieved the lowest error across all three metrics and was selected as the final forecasting model.

For the dashboard, **sMAPE** is used as the primary model-comparison visual because it expresses forecast error on a percentage scale and makes performance differences easier to interpret.

In [25]:
# Export the validated Phase 5 model-comparison results
# for the Revenue Forecast dashboard.

forecast_model_comparison = pd.DataFrame(
    {
        "model": [
            "Seasonal Naive (52-week)",
            "ARIMA(1,0,0)",
            "LightGBM",
            "ETS Damped Trend",
        ],
        "mae": [
            30106.90,
            56555.01,
            60541.55,
            79384.10,
        ],
        "rmse": [
            44972.90,
            77595.31,
            80225.80,
            101253.41,
        ],
        "smape_pct": [
            18.22,
            35.63,
            37.80,
            46.48,
        ],
    }
)


# Rank models by sMAPE:
# lower forecast error is better.
forecast_model_comparison["smape_rank"] = (
    forecast_model_comparison["smape_pct"]
    .rank(method="dense", ascending=True)
    .astype(int)
)


forecast_model_comparison["selected_model"] = (
    forecast_model_comparison["model"]
    .eq("Seasonal Naive (52-week)")
)


# Sort dashboard output from best to worst.
forecast_model_comparison = (
    forecast_model_comparison
    .sort_values(
        "smape_pct",
        ascending=True,
    )
    .reset_index(drop=True)
)


model_comparison_output_path = (
    DASHBOARD_DATA_DIR
    / "forecast_model_comparison.csv"
)


forecast_model_comparison.to_csv(
    model_comparison_output_path,
    index=False,
)


# -----------------------------
# Validation
# -----------------------------

assert len(forecast_model_comparison) == 4

assert (
    forecast_model_comparison.iloc[0]["model"]
    == "Seasonal Naive (52-week)"
)

assert np.isclose(
    forecast_model_comparison.iloc[0]["smape_pct"],
    18.22,
)

assert (
    forecast_model_comparison[
        "selected_model"
    ].sum()
    == 1
)


print(
    "Saved:",
    model_comparison_output_path,
)

print(
    "Models:",
    len(forecast_model_comparison),
)

print(
    "Selected model:",
    forecast_model_comparison.loc[
        forecast_model_comparison[
            "selected_model"
        ],
        "model",
    ].iloc[0],
)

print(
    "Best sMAPE:",
    f"{forecast_model_comparison['smape_pct'].min():.2f}%",
)


forecast_model_comparison

Saved: d:\Projects\retail-customer-retention\dashboard\data\forecast_model_comparison.csv
Models: 4
Selected model: Seasonal Naive (52-week)
Best sMAPE: 18.22%


,model,mae,rmse,smape_pct,smape_rank,selected_model
0,Seasonal Naive (52-week),30106.90,44972.90,18.22,1,True
1,"ARIMA(1,0,0)",56555.01,77595.31,35.63,2,False
2,LightGBM,60541.55,80225.80,37.80,3,False
3,ETS Damped Trend,79384.10,101253.41,46.48,4,False


#### Model Selection Result

The **52-week Seasonal Naive model** achieved the strongest out-of-sample forecasting performance, with:

- **MAE:** £30,106.90
- **RMSE:** £44,972.90
- **sMAPE:** 18.22%

It substantially outperformed the more complex alternatives:

- ARIMA: 35.63% sMAPE
- LightGBM: 37.80% sMAPE
- ETS Damped Trend: 46.48% sMAPE

This indicates that the strong annual seasonal pattern in the available weekly history was more useful than additional model complexity. Seasonal Naive is therefore retained as the final model for the Revenue Forecast dashboard.

### 7.8.4 Revenue Forecast KPI Summary

This export prepares the four headline metrics used at the top of the **Revenue Forecast** dashboard page:

- total forecast revenue over the next 13 weeks;
- average weekly forecast revenue;
- empirical weekly forecast error margin;
- selected forecasting model and its validation sMAPE.

These KPIs summarize both the expected revenue outlook and the reliability of the selected forecasting approach.

In [26]:
# Create the KPI summary used by the Revenue Forecast dashboard cards.

forecast_kpis = pd.DataFrame(
    {
        "forecast_total_13w": [
            weekly_revenue_forecast[
                "forecast_revenue"
            ].sum()
        ],
        "average_weekly_forecast": [
            weekly_revenue_forecast[
                "forecast_revenue"
            ].mean()
        ],
        "weekly_error_margin": [
            empirical_error_margin
        ],
        "selected_model": [
            forecast_model_comparison.loc[
                forecast_model_comparison[
                    "selected_model"
                ],
                "model",
            ].iloc[0]
        ],
        "selected_model_smape_pct": [
            forecast_model_comparison.loc[
                forecast_model_comparison[
                    "selected_model"
                ],
                "smape_pct",
            ].iloc[0]
        ],
        "forecast_start": [
            weekly_revenue_forecast[
                "week_end"
            ].min()
        ],
        "forecast_end": [
            weekly_revenue_forecast[
                "week_end"
            ].max()
        ],
        "forecast_weeks": [
            len(
                weekly_revenue_forecast
            )
        ],
    }
)


# Round dashboard-facing numeric values.
forecast_kpis[
    [
        "forecast_total_13w",
        "average_weekly_forecast",
        "weekly_error_margin",
        "selected_model_smape_pct",
    ]
] = (
    forecast_kpis[
        [
            "forecast_total_13w",
            "average_weekly_forecast",
            "weekly_error_margin",
            "selected_model_smape_pct",
        ]
    ]
    .round(2)
)


forecast_kpi_output_path = (
    DASHBOARD_DATA_DIR
    / "forecast_kpis.csv"
)


forecast_kpis.to_csv(
    forecast_kpi_output_path,
    index=False,
)


# -----------------------------
# Validation
# -----------------------------

assert len(forecast_kpis) == 1

assert np.isclose(
    forecast_kpis.loc[
        0,
        "forecast_total_13w",
    ],
    1433386.67,
)

assert np.isclose(
    forecast_kpis.loc[
        0,
        "average_weekly_forecast",
    ],
    110260.51,
)

assert np.isclose(
    forecast_kpis.loc[
        0,
        "weekly_error_margin",
    ],
    60478.76,
)

assert (
    forecast_kpis.loc[
        0,
        "selected_model"
    ]
    == "Seasonal Naive (52-week)"
)

assert np.isclose(
    forecast_kpis.loc[
        0,
        "selected_model_smape_pct",
    ],
    18.22,
)


print(
    "Saved:",
    forecast_kpi_output_path,
)

print(
    "13-week forecast:",
    f"£{forecast_kpis.loc[0, 'forecast_total_13w']:,.2f}",
)

print(
    "Average weekly forecast:",
    f"£{forecast_kpis.loc[0, 'average_weekly_forecast']:,.2f}",
)

print(
    "Weekly error margin:",
    f"±£{forecast_kpis.loc[0, 'weekly_error_margin']:,.2f}",
)

print(
    "Selected model:",
    forecast_kpis.loc[
        0,
        "selected_model"
    ],
)

print(
    "Selected model sMAPE:",
    f"{forecast_kpis.loc[0, 'selected_model_smape_pct']:.2f}%",
)


forecast_kpis

Saved: d:\Projects\retail-customer-retention\dashboard\data\forecast_kpis.csv
13-week forecast: £1,433,386.67
Average weekly forecast: £110,260.51
Weekly error margin: ±£60,478.76
Selected model: Seasonal Naive (52-week)
Selected model sMAPE: 18.22%


,forecast_total_13w,average_weekly_forecast,weekly_error_margin,selected_model,selected_model_smape_pct,forecast_start,forecast_end,forecast_weeks
0,1433386.67,110260.51,60478.76,Seasonal Naive (52-week),18.22,2011-12-16,2012-03-09,13


### 7.8.5 Historical and Forecast Revenue Timeline

For the main Revenue Forecast visual, the historical and forecast series are combined into a single dashboard-ready timeline.

The table preserves separate fields for:

- historical actual revenue;
- future point forecasts;
- empirical lower and upper planning bounds;
- the weekly error margin;
- the known zero-forecast rule artefact.

Keeping the historical and forecast values in separate columns allows Power BI to distinguish observed revenue from forward-looking estimates while using a common weekly date axis.

In [27]:
# Combine historical actuals and future forecasts into one
# dashboard-ready timeline for the main Page 3 visual.

history_timeline = (
    weekly_revenue_history
    .copy()
)

history_timeline["forecast_revenue"] = np.nan
history_timeline["error_margin"] = np.nan
history_timeline["lower_bound"] = np.nan
history_timeline["upper_bound"] = np.nan
history_timeline["zero_forecast_rule_artefact"] = False
history_timeline["period_type"] = "Historical"


forecast_timeline = (
    weekly_revenue_forecast
    .copy()
)

forecast_timeline["actual_revenue"] = np.nan
forecast_timeline["period_type"] = "Forecast"


revenue_forecast_timeline = pd.concat(
    [
        history_timeline,
        forecast_timeline,
    ],
    ignore_index=True,
)


revenue_forecast_timeline = (
    revenue_forecast_timeline[
        [
            "week_end",
            "period_type",
            "actual_revenue",
            "forecast_revenue",
            "error_margin",
            "lower_bound",
            "upper_bound",
            "zero_forecast_rule_artefact",
        ]
    ]
    .sort_values("week_end")
    .reset_index(drop=True)
)


timeline_output_path = (
    DASHBOARD_DATA_DIR
    / "revenue_forecast_timeline.csv"
)


revenue_forecast_timeline.to_csv(
    timeline_output_path,
    index=False,
)


# -----------------------------
# Validation
# -----------------------------

assert len(revenue_forecast_timeline) == 118

assert (
    revenue_forecast_timeline[
        "period_type"
    ].value_counts()["Historical"]
    == 105
)

assert (
    revenue_forecast_timeline[
        "period_type"
    ].value_counts()["Forecast"]
    == 13
)

assert (
    revenue_forecast_timeline["week_end"].min()
    == pd.Timestamp("2009-12-11")
)

assert (
    revenue_forecast_timeline["week_end"].max()
    == pd.Timestamp("2012-03-09")
)

assert (
    revenue_forecast_timeline[
        "zero_forecast_rule_artefact"
    ].sum()
    == 1
)


print(
    "Saved:",
    timeline_output_path,
)

print(
    "Rows:",
    len(revenue_forecast_timeline),
)

print(
    "Historical weeks:",
    (
        revenue_forecast_timeline[
            "period_type"
        ]
        .eq("Historical")
        .sum()
    ),
)

print(
    "Forecast weeks:",
    (
        revenue_forecast_timeline[
            "period_type"
        ]
        .eq("Forecast")
        .sum()
    ),
)

print(
    "Full timeline:",
    revenue_forecast_timeline[
        "week_end"
    ].min().date(),
    "to",
    revenue_forecast_timeline[
        "week_end"
    ].max().date(),
)


revenue_forecast_timeline.tail(18)

Saved: d:\Projects\retail-customer-retention\dashboard\data\revenue_forecast_timeline.csv
Rows: 118
Historical weeks: 105
Forecast weeks: 13
Full timeline: 2009-12-11 to 2012-03-09


,week_end,period_type,actual_revenue,forecast_revenue,error_margin,lower_bound,upper_bound,zero_forecast_rule_artefact
100,2011-11-11,Historical,273219.89,NaN,NaN,NaN,NaN,False
101,2011-11-18,Historical,268863.22,NaN,NaN,NaN,NaN,False
102,2011-11-25,Historical,256298.49,NaN,NaN,NaN,NaN,False
103,2011-12-02,Historical,243782.36,NaN,NaN,NaN,NaN,False
104,2011-12-09,Historical,424064.25,NaN,NaN,NaN,NaN,False
105,2011-12-16,Forecast,NaN,171530.20,60478.76,111051.44,232008.96,False
106,2011-12-23,Forecast,NaN,52456.86,60478.76,0.00,112935.62,False
107,2011-12-30,Forecast,NaN,0.00,60478.76,0.00,60478.76,True
108,2012-01-06,Forecast,NaN,98072.37,60478.76,37593.61,158551.13,False
109,2012-01-13,Forecast,NaN,161353.69,60478.76,100874.93,221832.45,False



### 7.8.6 Revenue Forecast Dashboard Export Validation

Before building the Power BI page, all Revenue Forecast dashboard exports are validated for expected row counts, date ranges, key forecast values, and model-selection results.

This ensures that the dashboard is built only from the validated Phase 5 forecasting outputs.

In [28]:
# Validate all Page 3 Revenue Forecast dashboard exports.

page3_expected_files = {
    "weekly_revenue_history.csv": 105,
    "weekly_revenue_forecast.csv": 13,
    "forecast_model_comparison.csv": 4,
    "forecast_kpis.csv": 1,
    "revenue_forecast_timeline.csv": 118,
}


print("PAGE 3 — REVENUE FORECAST EXPORT VALIDATION")
print("=" * 48)


for filename, expected_rows in page3_expected_files.items():

    path = DASHBOARD_DATA_DIR / filename

    assert path.exists(), f"Missing file: {filename}"

    exported_df = pd.read_csv(path)

    assert len(exported_df) == expected_rows, (
        f"{filename}: expected {expected_rows} rows, "
        f"found {len(exported_df)}"
    )

    print(
        f"PASS  {filename:<35} "
        f"{len(exported_df):>3} rows"
    )


# --------------------------------------------------
# Validate historical weekly series
# --------------------------------------------------

history_check = pd.read_csv(
    DASHBOARD_DATA_DIR
    / "weekly_revenue_history.csv",
    parse_dates=["week_end"],
)

assert history_check["week_end"].min() == pd.Timestamp(
    "2009-12-11"
)

assert history_check["week_end"].max() == pd.Timestamp(
    "2011-12-09"
)


# --------------------------------------------------
# Validate final 13-week forecast
# --------------------------------------------------

forecast_check = pd.read_csv(
    DASHBOARD_DATA_DIR
    / "weekly_revenue_forecast.csv",
    parse_dates=["week_end"],
)

assert forecast_check["week_end"].min() == pd.Timestamp(
    "2011-12-16"
)

assert forecast_check["week_end"].max() == pd.Timestamp(
    "2012-03-09"
)

assert np.isclose(
    forecast_check["forecast_revenue"].sum(),
    1433386.67,
)

assert np.isclose(
    forecast_check["forecast_revenue"].mean(),
    110260.51,
)

assert np.isclose(
    forecast_check["error_margin"].iloc[0],
    60478.76,
)

zero_artefact_rows = forecast_check.loc[
    forecast_check[
        "zero_forecast_rule_artefact"
    ]
]

assert len(zero_artefact_rows) == 1

assert (
    zero_artefact_rows.iloc[0]["week_end"]
    == pd.Timestamp("2011-12-30")
)

assert np.isclose(
    zero_artefact_rows.iloc[0]["forecast_revenue"],
    0,
)


# --------------------------------------------------
# Validate model comparison
# --------------------------------------------------

model_check = pd.read_csv(
    DASHBOARD_DATA_DIR
    / "forecast_model_comparison.csv"
)

selected_model_row = model_check.loc[
    model_check["selected_model"]
]

assert len(selected_model_row) == 1

assert (
    selected_model_row.iloc[0]["model"]
    == "Seasonal Naive (52-week)"
)

assert np.isclose(
    selected_model_row.iloc[0]["smape_pct"],
    18.22,
)


# --------------------------------------------------
# Validate KPI summary
# --------------------------------------------------

kpi_check = pd.read_csv(
    DASHBOARD_DATA_DIR
    / "forecast_kpis.csv"
)

assert np.isclose(
    kpi_check.loc[0, "forecast_total_13w"],
    1433386.67,
)

assert np.isclose(
    kpi_check.loc[0, "average_weekly_forecast"],
    110260.51,
)

assert np.isclose(
    kpi_check.loc[0, "weekly_error_margin"],
    60478.76,
)


print("\nAll Page 3 Revenue Forecast exports passed.")
print(
    "Forecast:",
    f"£{forecast_check['forecast_revenue'].sum():,.2f}"
)

print(
    "Average weekly:",
    f"£{forecast_check['forecast_revenue'].mean():,.2f}"
)

print(
    "Error margin:",
    f"±£{forecast_check['error_margin'].iloc[0]:,.2f}"
)

print(
    "Selected model:",
    selected_model_row.iloc[0]["model"],
)

print(
    "sMAPE:",
    f"{selected_model_row.iloc[0]['smape_pct']:.2f}%",
)

print(
    "Zero-forecast artefact:",
    zero_artefact_rows.iloc[0]["week_end"].date(),
)

PAGE 3 — REVENUE FORECAST EXPORT VALIDATION
PASS  weekly_revenue_history.csv          105 rows
PASS  weekly_revenue_forecast.csv          13 rows
PASS  forecast_model_comparison.csv         4 rows
PASS  forecast_kpis.csv                     1 rows
PASS  revenue_forecast_timeline.csv       118 rows

All Page 3 Revenue Forecast exports passed.
Forecast: £1,433,386.67
Average weekly: £110,260.51
Error margin: ±£60,478.76
Selected model: Seasonal Naive (52-week)
sMAPE: 18.22%
Zero-forecast artefact: 2011-12-30
